# Athena emulator parity — awswrangler write & catalog paths

Fifth notebook of the `athena_emulator` parity series: exercises awswrangler's *write* and *catalog* surface against the live stack — `wr.s3.to_parquet(dataset=True)` writes, named CTAS via `wr.athena.create_ctas_table`, `wr.athena.generate_create_query`, the `wr.catalog` partition helpers, `wr.s3.store_parquet_metadata`, and the delete flows that close the tutorial arc. Boundary probes measure `CREATE EXTERNAL TABLE`, `ALTER TABLE ... ADD PARTITION`, the Iceberg write path (`to_iceberg`/`delete_from_iceberg_table`), SSE-KMS encryption settings, and the out-of-scope Studio/session/capacity operations' shaped errors. Each probe records a measured `PASS`/`FAIL`/`GAP`; the session matrix persists to `PARITY.md`.

Stack (`docker-compose.yml`, network `mlops_net`): `athena` (emulator) · `trino` (query engine) · `moto` (S3/Glue) · `jupyterlab` (this notebook).

In [1]:
import os

# Compose sets these for in-container runs; the localhost fallbacks let the
# same notebook run against the published ports from the host.
os.environ.setdefault("AWS_ENDPOINT_URL", "http://localhost:5000")
os.environ.setdefault("AWS_ENDPOINT_URL_ATHENA", "http://localhost:5001")
os.environ.setdefault("AWS_ACCESS_KEY_ID", "test")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "test")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")

ATHENA_URL = os.environ["AWS_ENDPOINT_URL_ATHENA"]
EDGE_URL = os.environ["AWS_ENDPOINT_URL"]

print(f"athena → {ATHENA_URL}")
print(f"s3/glue → {EDGE_URL}")

athena → http://localhost:5001
s3/glue → http://localhost:5000


In [2]:
import time
import uuid

import awswrangler as wr
import boto3
import pandas as pd
from _evidence import persist, record, render
from _helpers import (
    delete_run_buckets,
    delete_run_databases,
    error_detail,
    object_keys_under,
    probe_outcome,
)
from botocore.exceptions import ClientError

athena = boto3.client("athena")
s3 = boto3.client("s3")
glue = boto3.client("glue")

assert athena.meta.endpoint_url.rstrip("/") == ATHENA_URL

RUN_SUFFIX = uuid.uuid4().hex[:8]
# Every resource this run creates carries the nb05- prefix (nb05_ for
# Glue databases and tables, where '-' is invalid) so cleanup can
# discover leftovers — its own and any failed run's — instead of
# tracking a list.
ATHENA_RESOURCE_PREFIX = "nb05-"
DATABASE_PREFIX = "nb05_"

bucket = f"{ATHENA_RESOURCE_PREFIX}{uuid.uuid4().hex[:12]}"
database = f"{DATABASE_PREFIX}{RUN_SUFFIX}"
orders_table = f"{DATABASE_PREFIX}orders_{RUN_SUFFIX}"
sales_table = f"{DATABASE_PREFIX}sales_{RUN_SUFFIX}"
meta_table = f"{DATABASE_PREFIX}meta_{RUN_SUFFIX}"
scratch_database = f"{DATABASE_PREFIX}scratch_{RUN_SUFFIX}"
results_prefix = f"s3://{bucket}/results/"

s3.create_bucket(Bucket=bucket)
glue.create_database(DatabaseInput={"Name": database})

{'ResponseMetadata': {'RequestId': 'D09MulmHnmg3FQuVe5o6q2eNGZMMnRI77EiAp8A9s27dgj64CYbo',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'server': 'Werkzeug/3.1.3 Python/3.13.9, amazon.com',
   'date': 'Wed, 07 Oct 2026 07:17:39 GMT, Wed, 07 Oct 2026 07:17:39 GMT',
   'x-amzn-requestid': 'D09MulmHnmg3FQuVe5o6q2eNGZMMnRI77EiAp8A9s27dgj64CYbo',
   'content-type': 'text/html; charset=utf-8',
   'content-length': '0',
   'access-control-allow-origin': '*',
   'connection': 'close'},
  'RetryAttempts': 0}}

## Helpers

In [3]:
def run_statement(sql: str) -> dict:
    """Submit ``sql`` and poll to a terminal state; returns the execution."""
    qid = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={"Database": database},
        ResultConfiguration={"OutputLocation": results_prefix},
    )["QueryExecutionId"]
    for _ in range(60):
        execution = athena.get_query_execution(QueryExecutionId=qid)[
            "QueryExecution"
        ]
        state = execution["Status"]["State"]
        if state in ("SUCCEEDED", "FAILED", "CANCELLED"):
            return execution
        time.sleep(1)
    raise TimeoutError(f"{qid} did not reach a terminal state")


def terminal_detail(outcome: dict | Exception) -> str:
    """State + reason for a finished execution, or the wire error."""
    if isinstance(outcome, Exception):
        return error_detail(outcome)
    state = outcome["Status"]["State"]
    reason = outcome["Status"].get("StateChangeReason", "")
    detail = (
        reason.splitlines()[0]
        if reason
        else outcome.get("QueryExecutionId", "-")
    )
    return f"{state}: {detail[:80]}"


def cleanup_run_resources() -> None:
    """Remove everything this run (or a failed earlier run) left behind."""
    delete_run_buckets(s3, ATHENA_RESOURCE_PREFIX)
    delete_run_databases(glue, DATABASE_PREFIX)

## Dataset — `wr.s3.to_parquet(dataset=True)`

The write half of the read-path flow nb04 exercised: one plain table and one partitioned by `region`. `dataset=True` registers the table (and its Hive-style `region=…/` partition prefixes) in the Glue catalog so Trino can resolve it.

In [4]:
orders = pd.DataFrame(
    {"id": [1, 2, 3], "item": ["a", "b", "c"], "amount": [9.5, 1.0, 4.25]}
)
sales = pd.DataFrame(
    {"quantity": [1, 2], "amount": [9.5, 4.0], "region": ["EU", "US"]}
)
wr.s3.to_parquet(
    df=orders,
    path=f"s3://{bucket}/datasets/orders/",
    dataset=True,
    database=database,
    table=orders_table,
)
wr.s3.to_parquet(
    df=sales,
    path=f"s3://{bucket}/datasets/sales/",
    dataset=True,
    database=database,
    table=sales_table,
    partition_cols=["region"],
)
seed_partitions = wr.catalog.get_partitions(
    database=database, table=sales_table
)
assert wr.catalog.does_table_exist(database=database, table=orders_table)
record(
    "wr.s3.to_parquet(dataset=True) registers tables + partitions",
    "PASS",
    f"{orders_table} + {sales_table} registered; sales partitions={list(seed_partitions.values())}",
)
wr.catalog.tables(database=database)

,Database,Table,Description,TableType,Columns,Partitions
0,nb05_ecdee0a5,nb05_orders_ecdee0a5,,EXTERNAL_TABLE,"id, item, amount",
1,nb05_ecdee0a5,nb05_sales_ecdee0a5,,EXTERNAL_TABLE,"quantity, amount",region


## `wr.athena.create_ctas_table` — a named CTAS table

Unlike the throwaway CTAS `read_sql_query(ctas_approach=True)` issues, `create_ctas_table` names the target table and leaves it registered. Wrangler emits Trino's `CREATE TABLE … WITH(external_location=…, format='PARQUET') AS <sql>` spelling, so the parquet data lands under `<s3_output>/<ctas_table>/`.

In [5]:
ctas_table = f"{DATABASE_PREFIX}ctas_{RUN_SUFFIX}"
started = time.monotonic()
ctas_result = probe_outcome(
    lambda: wr.athena.create_ctas_table(
        sql=f'SELECT * FROM "{database}"."{orders_table}" WHERE amount > 2',
        database=database,
        ctas_database=database,
        ctas_table=ctas_table,
        s3_output=f"s3://{bucket}/ctas-out/",
        wait=True,
    )
)
if isinstance(ctas_result, dict):
    ctas_qid = ctas_result["ctas_query_metadata"].execution_id
    ctas_frame = wr.athena.read_sql_query(
        f'SELECT * FROM "{database}"."{ctas_table}"',
        database=database,
        ctas_approach=False,
        s3_output=results_prefix,
    )
    data_objects = object_keys_under(s3, bucket, f"ctas-out/{ctas_table}/")
    assert ctas_frame["id"].tolist() == [1, 3]
    assert data_objects
    record(
        "wr.athena.create_ctas_table (named CTAS → Glue table + parquet)",
        "PASS",
        f"{ctas_qid} read back {ctas_frame.shape}; {len(data_objects)} object(s) at external_location; manifest="
        f"{ctas_result['ctas_query_metadata'].manifest_location.split('/')[-1]}",
        (time.monotonic() - started) * 1000,
    )
    print(ctas_frame)
else:
    record(
        "wr.athena.create_ctas_table (named CTAS → Glue table + parquet)",
        "GAP",
        error_detail(ctas_result),
    )

   id item  amount
0   1    a    9.50
1   3    c    4.25


## `wr.athena.generate_create_query` — DDL from the catalog

Reads the Glue `Table` definition and emits the `CREATE EXTERNAL TABLE` statement that would recreate it — a pure catalog read, useful for the `CREATE EXTERNAL TABLE` probe below since the text wrangler generates is exactly the Athena DDL spelling.

In [6]:
generated_ddl = probe_outcome(
    lambda: wr.athena.generate_create_query(
        table=sales_table, database=database
    )
)
if isinstance(generated_ddl, str):
    record(
        "generate_create_query emits CREATE EXTERNAL TABLE DDL",
        "PASS",
        f"{len(generated_ddl)} chars; external={'CREATE EXTERNAL TABLE' in generated_ddl} partitioned={'PARTITIONED BY' in generated_ddl}",
    )
    print(generated_ddl)
else:
    record(
        "generate_create_query emits CREATE EXTERNAL TABLE DDL",
        "GAP",
        error_detail(generated_ddl),
    )
    generated_ddl = None

CREATE EXTERNAL TABLE `nb05_sales_ecdee0a5`(
  `quantity` bigint, 
  `amount` double)
PARTITIONED BY ( 
  `region` string)
ROW FORMAT SERDE 
  'org.apache.hadoop.hive.ql.io.parquet.serde.ParquetHiveSerDe' 
STORED AS INPUTFORMAT 
  'org.apache.hadoop.hive.ql.io.parquet.MapredParquetInputFormat' 
OUTPUTFORMAT 
  'org.apache.hadoop.hive.ql.io.parquet.MapredParquetOutputFormat'
LOCATION
  's3://nb05-c6a8f99f9cf5/datasets/sales/'
TBLPROPERTIES (
  'classification'='parquet', 
  'compressionType'='snappy', 
  'typeOfData'='file', 
  'projection.enabled'='false')


## `wr.catalog` partition helpers — add / list / delete

Files written outside the `dataset=True` path never reach the catalog; `add_parquet_partitions` registers a staged partition location explicitly, `get_partitions` reads the registry, and `delete_partitions` unregisters without touching the objects.

In [7]:
ap_sales = pd.DataFrame({"quantity": [5], "amount": [2.5], "region": ["AP"]})
ap_location = f"s3://{bucket}/datasets/sales/region=AP/"
wr.s3.to_parquet(
    df=ap_sales,
    path=f"{ap_location}part-0.parquet",
    dataset=False,
)
wr.catalog.add_parquet_partitions(
    database=database,
    table=sales_table,
    partitions_values={ap_location: ["AP"]},
)
after_add = wr.catalog.get_partitions(database=database, table=sales_table)
assert ["AP"] in after_add.values()
record(
    "wr.catalog.add_parquet_partitions registers a staged partition",
    "PASS",
    f"{len(after_add)} partitions after add: {list(after_add.values())}",
)
print(after_add)

{'s3://nb05-c6a8f99f9cf5/datasets/sales/region=EU/': ['EU'], 's3://nb05-c6a8f99f9cf5/datasets/sales/region=US/': ['US'], 's3://nb05-c6a8f99f9cf5/datasets/sales/region=AP/': ['AP']}


In [8]:
wr.catalog.delete_partitions(
    table=sales_table,
    database=database,
    partitions_values=[["AP"]],
)
after_delete = wr.catalog.get_partitions(database=database, table=sales_table)
assert ["AP"] not in after_delete.values()
record(
    "wr.catalog.delete_partitions unregisters a partition",
    "PASS",
    f"{len(after_delete)} partitions after delete: {list(after_delete.values())}",
)
print(after_delete)

{'s3://nb05-c6a8f99f9cf5/datasets/sales/region=EU/': ['EU'], 's3://nb05-c6a8f99f9cf5/datasets/sales/region=US/': ['US']}


## `wr.s3.store_parquet_metadata` — register an existing prefix

The inverse of `to_parquet(dataset=True)`: parquet files already on S3 get their schema inferred and the table + Hive-style partitions registered in one call — no `MSCK REPAIR TABLE` needed.

In [9]:
meta_prefix = f"s3://{bucket}/meta/sales/"
wr.s3.to_parquet(
    df=sales,
    path=meta_prefix,
    dataset=True,
    partition_cols=["region"],
)
meta_outcome = probe_outcome(
    lambda: wr.s3.store_parquet_metadata(
        path=meta_prefix,
        database=database,
        table=meta_table,
        dataset=True,
    )
)
if isinstance(meta_outcome, tuple):
    columns_types, partitions_types, partitions_values = meta_outcome
    meta_partitions = wr.catalog.get_partitions(
        database=database, table=meta_table
    )
    assert wr.catalog.does_table_exist(database=database, table=meta_table)
    record(
        "wr.s3.store_parquet_metadata registers inferred schema + partitions",
        "PASS",
        f"{meta_table} columns={sorted(columns_types)} partition_types={partitions_types} values={list(meta_partitions.values())}",
    )
    wr.catalog.table(database=database, table=meta_table)
else:
    record(
        "wr.s3.store_parquet_metadata registers inferred schema + partitions",
        "GAP",
        error_detail(meta_outcome),
    )

## Delete flows — objects, partitions, tables, databases

The tutorial teardown arc, exercised on dedicated resources: `wr.s3.delete_objects` removes a prefix, `delete_all_partitions` unregisters every partition, `delete_table_if_exists` drops the catalog entry, `delete_database` removes an empty database.

In [10]:
wr.catalog.create_database(scratch_database)
wr.s3.delete_objects(path=meta_prefix)
wr.catalog.delete_all_partitions(table=meta_table, database=database)
remaining_partitions = wr.catalog.get_partitions(
    database=database, table=meta_table
)
wr.catalog.delete_table_if_exists(database=database, table=meta_table)
wr.catalog.delete_database(scratch_database)
remaining_meta = object_keys_under(s3, bucket, "meta/sales/")
remaining_databases = {d["Name"] for d in glue.get_databases()["DatabaseList"]}
assert remaining_meta == []
assert remaining_partitions == {}
assert not wr.catalog.does_table_exist(database=database, table=meta_table)
assert scratch_database not in remaining_databases
record(
    "wr.s3.delete_objects removes a staged prefix",
    "PASS",
    f"{meta_prefix} emptied",
)
record(
    "wr.catalog.delete_all_partitions empties the registry",
    "PASS",
    f"{meta_table} has 0 partitions",
)
record(
    "wr.catalog.delete_table_if_exists drops the table",
    "PASS",
    f"{meta_table} absent from the catalog",
)
record(
    "wr.catalog.delete_database removes an empty database",
    "PASS",
    f"{scratch_database} absent from get_databases",
)

Evidence(feature='wr.catalog.delete_database removes an empty database', status='PASS', detail='nb05_scratch_ecdee0a5 absent from get_databases', latency_ms=None)

## Boundary probe — `CREATE EXTERNAL TABLE`

Athena's DDL spelling carries `EXTERNAL` + `STORED AS` + `TBLPROPERTIES`; Trino's hive connector accepts only `CREATE TABLE … WITH(…)`. The measured outcome — success or the wire error — is recorded as-is.

In [11]:
feature = "CREATE EXTERNAL TABLE via StartQueryExecution (Athena DDL)"
external_ddl = (
    f"CREATE EXTERNAL TABLE {DATABASE_PREFIX}ext_{RUN_SUFFIX} "
    "(id bigint, item string)\n"
    "STORED AS PARQUET\n"
    f"LOCATION 's3://{bucket}/ext/'"
)
external_outcome = probe_outcome(lambda: run_statement(external_ddl))
external_ok = (
    isinstance(external_outcome, dict)
    and external_outcome["Status"]["State"] == "SUCCEEDED"
)
record(
    feature,
    "PASS" if external_ok else "GAP",
    terminal_detail(external_outcome),
)

Evidence(feature='CREATE EXTERNAL TABLE via StartQueryExecution (Athena DDL)', status='PASS', detail='SUCCEEDED: 3ee5f20f-36ec-4312-a229-753ff85f8e61', latency_ms=None)

## Boundary probe — `ALTER TABLE … ADD PARTITION`

Athena registers partitions with `ALTER TABLE t ADD PARTITION (col='v') LOCATION 's3://…'`; Trino's hive connector exposes partition registration through the `system.register_partition` procedure instead of the grammar form.

In [12]:
feature = "ALTER TABLE … ADD PARTITION (Athena DDL)"
alter_sql = (
    f"ALTER TABLE {sales_table} ADD IF NOT EXISTS "
    "PARTITION (region='AP') "
    f"LOCATION 's3://{bucket}/datasets/sales/region=AP/'"
)
alter_outcome = probe_outcome(lambda: run_statement(alter_sql))
alter_ok = (
    isinstance(alter_outcome, dict)
    and alter_outcome["Status"]["State"] == "SUCCEEDED"
)
record(
    feature,
    "PASS" if alter_ok else "GAP",
    terminal_detail(alter_outcome),
)

Evidence(feature='ALTER TABLE … ADD PARTITION (Athena DDL)', status='PASS', detail='SUCCEEDED: b51cd8d4-5576-4dc9-a8ae-f3ec7339e407', latency_ms=None)

## Boundary probe — Iceberg writes

`wr.athena.to_iceberg` emits a backtick-quoted `CREATE TABLE … TBLPROPERTIES('table_type'='ICEBERG')` then `INSERT INTO … SELECT`; `delete_from_iceberg_table` stages a temp table then issues `DELETE FROM`. Both lean on Athena's Iceberg surface, which needs an Iceberg connector the hive catalog doesn't provide.

In [13]:
feature = "wr.athena.to_iceberg (Iceberg table write)"
iceberg_table = f"{DATABASE_PREFIX}ice_{RUN_SUFFIX}"
iceberg_df = pd.DataFrame({"id": [1, 2], "v": ["x", "y"]})
iceberg_outcome = probe_outcome(
    lambda: wr.athena.to_iceberg(
        df=iceberg_df,
        database=database,
        table=iceberg_table,
        temp_path=f"s3://{bucket}/iceberg-tmp/",
        table_location=f"s3://{bucket}/iceberg/{iceberg_table}/",
        s3_output=results_prefix,
    )
)
iceberg_created = iceberg_outcome is None and (
    wr.catalog.does_table_exist(database=database, table=iceberg_table)
)
record(
    feature,
    "PASS" if iceberg_created else "GAP",
    "created" if iceberg_outcome is None else error_detail(iceberg_outcome),
)

Evidence(feature='wr.athena.to_iceberg (Iceberg table write)', status='PASS', detail='created', latency_ms=None)

In [14]:
feature = "wr.athena.delete_from_iceberg_table (row delete)"
delete_outcome = probe_outcome(
    lambda: wr.athena.delete_from_iceberg_table(
        df=pd.DataFrame({"id": [1]}),
        database=database,
        table=iceberg_table,
        merge_cols=["id"],
        temp_path=f"s3://{bucket}/iceberg-del-tmp/",
        s3_output=results_prefix,
    )
)
record(
    feature,
    "PASS" if delete_outcome is None else "GAP",
    "deleted" if delete_outcome is None else error_detail(delete_outcome),
)

Evidence(feature='wr.athena.delete_from_iceberg_table (row delete)', status='PASS', detail='deleted', latency_ms=None)

## Encryption / KMS settings

Two surfaces take encryption settings: the `EncryptionConfiguration` on a query's `ResultConfiguration` (`create_ctas_table` maps `encryption`/`kms_key` onto it — real Athena echoes it back on `GetQueryExecution`) and the S3 write headers (`s3_additional_kwargs`).

In [15]:
feature = "SSE_KMS EncryptionConfiguration on CTAS"
enc_table = f"{DATABASE_PREFIX}enc_{RUN_SUFFIX}"
kms_key = (
    "arn:aws:kms:us-east-1:123456789012:key/"
    "00000000-0000-0000-0000-000000000000"
)
enc_outcome = probe_outcome(
    lambda: wr.athena.create_ctas_table(
        sql="SELECT 1 AS n",
        database=database,
        ctas_database=database,
        ctas_table=enc_table,
        s3_output=results_prefix,
        encryption="SSE_KMS",
        kms_key=kms_key,
        wait=True,
    )
)
if isinstance(enc_outcome, dict):
    echo = (
        enc_outcome["ctas_query_metadata"]
        .raw_payload["ResultConfiguration"]
        .get("EncryptionConfiguration")
    )
    record(
        feature,
        "PASS"
        if echo == {"EncryptionOption": "SSE_KMS", "KmsKey": kms_key}
        else "GAP",
        f"EncryptionConfiguration={echo}",
    )
else:
    record(feature, "GAP", error_detail(enc_outcome))

In [16]:
feature = "wr.s3.to_parquet SSE-KMS write headers"
kms_outcome = probe_outcome(
    lambda: wr.s3.to_parquet(
        df=orders,
        path=f"s3://{bucket}/kms/orders.parquet",
        dataset=False,
        s3_additional_kwargs={
            "ServerSideEncryption": "aws:kms",
            "SSEKMSKeyId": kms_key,
        },
    )
)
kms_objects = object_keys_under(s3, bucket, "kms/")
record(
    feature,
    "PASS"
    if not isinstance(kms_outcome, Exception) and kms_objects
    else "GAP",
    f"objects={kms_objects}"
    if not isinstance(kms_outcome, Exception)
    else error_detail(kms_outcome),
)

Evidence(feature='wr.s3.to_parquet SSE-KMS write headers', status='PASS', detail="objects=['kms/orders.parquet']", latency_ms=None)

## Out-of-scope operations — shaped errors

The Studio/notebook/session/calculation/capacity surface is declared out of scope (ADR-0004): every such operation is present in the dispatcher but answers a botocore-parseable `InvalidRequestException` 400 naming the operation — never an unshaped 404/500.

In [17]:
oos_calls = {
    "create_capacity_reservation": lambda: athena.create_capacity_reservation(
        Name=f"nb05-{RUN_SUFFIX}", TargetDpus=4
    ),
    "list_capacity_reservations": athena.list_capacity_reservations,
    "create_notebook": lambda: athena.create_notebook(
        WorkGroup="primary", Name=f"nb05-{RUN_SUFFIX}"
    ),
    "list_sessions": lambda: athena.list_sessions(WorkGroup="primary"),
}
for op_name, op_call in oos_calls.items():
    outcome = probe_outcome(op_call)
    shaped = (
        isinstance(outcome, ClientError)
        and outcome.response["Error"]["Code"] == "InvalidRequestException"
        and outcome.response["ResponseMetadata"]["HTTPStatusCode"] == 400
    )
    record(
        f"out-of-scope {op_name} → shaped InvalidRequestException",
        "PASS" if shaped else "FAIL",
        error_detail(outcome)
        if isinstance(outcome, Exception)
        else "unexpected success",
    )

## Session matrix

In [18]:
print(render())
persist("05_wrangler_write_catalog_and_gaps")

| feature | status | detail | latency_ms |
|---|---|---|---|
| wr.s3.to_parquet(dataset=True) registers tables + partitions | PASS | nb05_orders_ecdee0a5 + nb05_sales_ecdee0a5 registered; sales partitions=[['EU'], ['US']] |  |
| wr.athena.create_ctas_table (named CTAS → Glue table + parquet) | PASS | f25584e1-3778-4e31-a25a-116fba40982c read back (2, 3); 1 object(s) at external_location; manifest=f25584e1-3778-4e31-a25a-116fba40982c-manifest.csv | 2514.48 |
| generate_create_query emits CREATE EXTERNAL TABLE DDL | PASS | 562 chars; external=True partitioned=True |  |
| wr.catalog.add_parquet_partitions registers a staged partition | PASS | 3 partitions after add: [['EU'], ['US'], ['AP']] |  |
| wr.catalog.delete_partitions unregisters a partition | PASS | 2 partitions after delete: [['EU'], ['US']] |  |
| wr.s3.store_parquet_metadata registers inferred schema + partitions | PASS | nb05_meta_ecdee0a5 columns=['amount', 'quantity'] partition_types={'region': 'string'} values=[['EU'], ['U

PosixPath('/home/gabriel-menezes/Documents/repos/mlops/projects/athena_emulator/parity/05_wrangler_write_catalog_and_gaps.md')

In [19]:
cleanup_run_resources()
print("cleaned: nb05 buckets + glue databases (incl. tables) removed")

cleaned: nb05 buckets + glue databases (incl. tables) removed
